# Outbound Auth

Outbound Auth를 사용하면 에이전트와 AgentCore Gateway가 Inbound Auth에서 인증 및 권한 부여된 사용자를 대신해 AWS 리소스와 서드 파티 서비스에 안전하게 액세스할 수 있습니다. AWS 리소스 또는 서드 파티 서비스와 권한 부여를 통합하려면 Inbound Auth와 Outbound Auth를 모두 구성해야 합니다.

AgentCore Identity가 지원하는 최소 필요 액세스와 안전한 권한 위임을 통해 에이전트는 AWS 리소스 및 GitHub, Google, Salesforce, Slack 같은 서드 파티 도구에 원활하고 안전하게 액세스할 수 있습니다. 사전에 사용자의 동의를 받았다면 에이전트는 사용자를 대신하거나 독립적으로 이러한 서비스에서 작업을 수행할 수 있습니다. 또한 안전한 token vault를 사용해 반복적인 동의 요청을 줄이고 간소화된 AI 에이전트 경험을 만들 수 있습니다.

## Outbound Auth 구성

먼저 서드 파티 provider에 클라이언트 애플리케이션을 등록한 다음 Outbound Auth를 생성합니다. AWS 리소스, 서드 파티 서비스 또는 AgentCore Gateway 대상에 대한 액세스를 검증할 방법을 지정합니다. OAuth 2LO/3LO 또는 API key를 사용할 수 있습니다. OAuth를 사용할 때는 AgentCore Identity가 제공하는 provider를 선택하고 해당 구성 정보를 입력하거나, custom provider의 세부 정보를 직접 제공할 수 있습니다. 

사용자가 AWS 리소스, 서드 파티 서비스 또는 AgentCore Gateway 대상에 액세스하려고 하면 Outbound Auth가 Inbound Auth에서 제공한 access token의 유효성을 확인하고, 유효한 경우 리소스 액세스를 허용합니다.

<div style="text-align:center">
    <img src="images/outbound_auth.png" width="90%"/>
</div>

## Resource credential provider

Resource credential provider는 에이전트 코드가 downstream resource server(예: Google, GitHub)의 자격 증명을 가져와 Gmail 이메일을 조회하거나 Google Calendar에 회의를 추가하는 등의 작업에 액세스할 때 사용하는 구성 요소입니다. 최종 사용자, 에이전트 코드, 외부 authorization server 사이의 2LO 및 3LO OAuth2 오케스트레이션 흐름을 에이전트 개발자가 직접 구현해야 하는 부담을 줄여 줍니다. AgentCore는 custom OAuth2 credential provider와 함께 Google, GitHub, Slack, Salesforce 등 authorization server endpoint와 provider별 파라미터가 미리 채워진 built-in provider 목록을 제공합니다.
  

Bedrock AgentCore Identity는 에이전트 개발자가 OAuth2 또는 API key를 지원하는 외부 리소스에 인증할 수 있도록 OAuth2 및 API Key Credential Provider를 제공합니다. 다음 예제에서는 API Key credential provider를 구성합니다. 이후 에이전트는 이 provider에서 에이전트 작업에 필요한 API key를 가져올 수 있습니다. 다른 credential provider는 관련 문서를 참조하세요.

 ### Resource credential provider 생성

 다음은 API Key resource credential provider를 생성하는 예제입니다.

```
from bedrock_agentcore.services.identity import IdentityClient
identity_client = IdentityClient(region="us-west-2")

api_key_provider = identity_client.create_api_key_credential_provider({
    "name": "APIKey-provider-name",
    "apiKey": "<my-api-key>" # OpenAI 등 외부 애플리케이션 공급업체에서 받은 API key로 교체
})
print(api_key_provider)
```

 ### Resource credential provider에서 access token 또는 API key 가져오기

다음은 API Key credential provider에서 API key를 가져오는 예제입니다. 에이전트는 이 API key를 사용해 LLM 또는 API key 구성을 사용하는 다른 서비스와 상호 작용할 수 있습니다. credential provider에서 `access_token`이나 API key 같은 자격 증명을 가져오려면 다음과 같이 함수에 decorator를 적용합니다.

```
import asyncio
from bedrock_agentcore.identity.auth import requires_access_token, requires_api_key

@requires_api_key(
    provider_name="APIKey-provider" # 직접 생성한 credential provider 이름으로 교체
)
async def need_api_key(*, api_key: str):
    print(f'received api key for async func: {api_key}')

await need_api_key(api_key="")
```

다음은 `@require_access_token` decorator에서 사용할 수 있는 파라미터입니다.


| 파라미터 이름       | 설명                                                                     |
|:--------------------|:-------------------------------------------------------------------------|
| provider_name       | credential provider 이름                                                 |
| into                | 토큰을 주입할 파라미터 이름                                              |
| scopes              | 요청할 OAuth2 scope                                                      |
| on_auth_url	      | authorization URL 처리용 callback                                        |
| auth_flow           | 인증 흐름 유형("M2M" 또는 "USER_FEDERATION")                            |
| callback_url        | OAuth2 callback URL                                                      |
| force_authentication| 재인증 강제                                                               |
| token_poller        | custom token poller 구현                                                  |

		


# OpenAI 모델을 사용하는 Strands Agents를 Amazon Bedrock AgentCore Runtime에 호스팅

## 개요


이 튜토리얼에서는 01-AgentCore-runtime에서 배포한 OpenAI 모델 기반 에이전트를 수정하고 API Key credential provider를 사용하는 Outbound Auth를 구성합니다. OpenAI key를 저장할 API Key credential provider를 설정한 다음, 이 key를 사용하도록 에이전트 코드를 수정합니다.

### 튜토리얼 아키텍처

<div style="text-align:center">
    <img src="images/outbound_auth_api.png" width="90%"/>
</div>


### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                |
|:--------------------|:-------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                   |
| 에이전트 유형       | 단일                                                                     |
| Agentic Framework   | Strands Agents                                                           |
| LLM model           | GPT 4.1 mini                                                             |
| 튜토리얼 구성 요소  | AgentCore Runtime에 에이전트 호스팅, Strands Agent 및 OpenAI 모델 사용   |
| 튜토리얼 분야       | 산업 공통                                                                |
| 예제 난이도         | 쉬움                                                                     |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                              |
| Credential Provider | 유형: API Key                                                            |


### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* OpenAI 모델 사용
* Strands Agents 사용
* API Key credential provider를 이용한 AgentCore Outbound Auth 사용


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker
* OpenAI API key

OpenAI API key를 얻는 방법:
- OpenAI: [OpenAI API key](https://help.openai.com/en/articles/4936850-where-do-i-find-my-openai-api-key)
- Azure OpenAI: [Azure OpenAI 리소스 생성 및 key 확인](https://learn.microsoft.com/azure/ai-services/openai/how-to/create-resource?tabs=azure-portal)

실행 전에 환경 변수를 설정하세요.
- OpenAI:
  - `OPENAI_API_KEY`
- Azure OpenAI:
  - `AZURE_OPENAI_API_KEY`
  - `AZURE_OPENAI_ENDPOINT`
  - `AZURE_OPENAI_API_VERSION`(예: `2024-02-15-preview`)
  - `AZURE_OPENAI_DEPLOYMENT`(배포한 모델 이름)

선택적 provider 전환:
- `OPENAI_PROVIDER` = `openai`(기본값) 또는 `azure`

참고:
- secret을 하드코딩하지 마세요. AgentCore Identity의 credential provider를 사용해 런타임에 API key를 저장하고 가져오며, key를 정기적으로 교체하세요.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## 에이전트 생성 및 로컬 실험

AgentCore Runtime에 에이전트를 배포하기 전에 로컬에서 개발하고 실행하며 실험해 보겠습니다.

프로덕션 에이전트 애플리케이션에서는 에이전트 생성 과정과 호출 과정을 분리해야 합니다. AgentCore Runtime에서는 에이전트 호출 부분에 `@app.entrypoint` decorator를 적용하여 Runtime의 엔트리포인트로 사용합니다. 먼저 실험 단계에서 에이전트를 개발하는 방법을 살펴보겠습니다.

아키텍처는 다음과 같습니다.

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [ ]:
%%writefile strands_agents_openai.py
from strands import Agent, tool
from strands_tools import calculator # calculator 도구 가져오기
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os

## 아래 구성을 Azure API Key 정보로 업데이트
os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# 사용자 지정 도구 생성
@tool
def weather():
    """ Get weather """ # 예제용 구현
    return "sunny"

model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)


agent = Agent(
    model=litellm_model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_open_ai(payload):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_open_ai(json.loads(args.payload))
    print(response)

#### 로컬 에이전트 호출

In [ ]:
!python strands_agents_openai.py '{"prompt": "What is the weather now?"}'

## Resource Credential Provider 생성

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient

from boto3.session import Session
import boto3

boto_session = Session()
region = boto_session.region_name

# API Key Provider 구성
identity_client = IdentityClient(region=region)

api_key_provider = identity_client.create_api_key_credential_provider(
    {
        "name": "openai-apikey-provider",
        "apiKey": "<YOUR_API_KEY>",  # OpenAI 등 외부 애플리케이션 공급업체에서 받은 API key로 교체
    }
)
print(api_key_provider)

## AgentCore Runtime 배포 및 Resource Credential Provider 사용을 위한 에이전트 준비

이제 에이전트를 AgentCore Runtime에 배포하겠습니다. 다음 작업이 필요합니다.
* `from bedrock_agentcore.runtime import BedrockAgentCoreApp`으로 Runtime App 가져오기
* 코드에서 `app = BedrockAgentCoreApp()`으로 App 초기화
* 호출 함수에 `@app.entrypoint` decorator 적용
* `app.run()`으로 AgentCore Runtime이 에이전트 실행을 제어하도록 구성
* 앞 단계에서 생성한 resource credential provider에서 OpenAI key 가져오기

### OpenAI 모델을 사용하는 Strands Agents
GPT 4.1 mini 모델을 사용하는 Strands Agent부터 시작하겠습니다. 다른 모델도 동일한 방식으로 작동합니다.

In [ ]:
%%writefile strands_agents_openai.py
import asyncio
from bedrock_agentcore.identity.auth import requires_access_token, requires_api_key
from strands import Agent, tool
from strands_tools import calculator 
import argparse
import json
from strands.models.litellm import LiteLLMModel
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp

AZURE_API_KEY_FROM_CREDS_PROVIDER = ""


@requires_api_key(
    provider_name="openai-apikey-provider" # 직접 생성한 credential provider 이름으로 교체
)
async def need_api_key(*, api_key: str):
    global AZURE_API_KEY_FROM_CREDS_PROVIDER
    print(f'received api key for async func: {api_key}')
    AZURE_API_KEY_FROM_CREDS_PROVIDER = api_key

# 모듈 수준에서는 빈 값을 출력하지 않고 엔트리포인트 함수에서 출력

app = BedrockAgentCoreApp()

# API key는 엔트리포인트 함수에서 동적으로 설정
# 아래 구성을 Azure API Key 정보로 업데이트
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

# 사용자 지정 도구 생성
@tool
def weather():
    """ Get weather """ # 예제용 구현
    return "sunny"

# 전역 에이전트 변수
agent = None

@app.entrypoint
async def strands_agent_open_ai(payload):
    """
    페이로드로 에이전트를 호출합니다.
    """
    global AZURE_API_KEY_FROM_CREDS_PROVIDER, agent
    
    print(f"Entrypoint called with AZURE_API_KEY_FROM_CREDS_PROVIDER: '{AZURE_API_KEY_FROM_CREDS_PROVIDER}'")
    
    # 아직 가져오지 않았다면 API key 가져오기
    if not AZURE_API_KEY_FROM_CREDS_PROVIDER:
        print("Attempting to retrieve API key...")
        try:
            await need_api_key(api_key="")
            print(f"API key retrieved: '{AZURE_API_KEY_FROM_CREDS_PROVIDER}'")
            os.environ["AZURE_API_KEY"] = AZURE_API_KEY_FROM_CREDS_PROVIDER
            print("Environment variable AZURE_API_KEY set")
        except Exception as e:
            print(f"Error retrieving API key: {e}")
            raise
    else:
        print("API key already available")
    
    # API key 설정 후 에이전트 초기화
    if agent is None:
        print("Initializing agent with API key...")
        model = "azure/gpt-4.1-mini"
        litellm_model = LiteLLMModel(
            model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
        )
        
        agent = Agent(
            model=litellm_model,
            tools=[calculator, weather],
            system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
        )
        print("Agent initialized successfully")
    
    user_input = payload.get("prompt")
    print(f"User input: {user_input}")
    
    try:
        response = agent(user_input)
        print(f"Agent response: {response}")
        return response.message['content'][0]['text']
    except Exception as e:
        print(f"Error in agent processing: {e}")
        raise

if __name__ == "__main__":
    app.run()


## 내부에서는 어떤 작업이 수행되나요?

`BedrockAgentCoreApp`을 사용하면 다음 작업이 자동으로 수행됩니다.

* 포트 8080에서 수신 대기하는 HTTP server 생성
* 에이전트 요청을 처리하는 필수 `/invocations` endpoint 구현
* 상태 확인용 `/ping` endpoint 구현(비동기 에이전트에 매우 중요)
* 적절한 content type 및 응답 형식 처리
* AWS 표준에 따른 오류 처리 관리

## AgentCore Runtime에 에이전트 배포

`CreateAgentRuntime` 작업은 컨테이너 이미지, 환경 변수, 암호화 설정 등 폭넓은 구성 옵션을 지원합니다. 프로토콜 설정(HTTP, MCP)과 권한 부여 메커니즘도 구성하여 클라이언트가 에이전트와 통신하는 방식을 제어할 수 있습니다. 

**참고:** 운영 환경에서는 코드를 컨테이너로 패키징하고 CI/CD 파이프라인과 IaC를 사용해 ECR에 푸시하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK로 아티팩트를 간편하게 패키징하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포 구성

이제 starter toolkit을 사용해 엔트리포인트, 앞에서 생성한 실행 역할, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 시작 시 Amazon ECR 리포지토리를 자동 생성하도록 starter toolkit도 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import json

boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()
agent_name = "strands_agents_openai"

response = agentcore_runtime.configure(
    entrypoint="strands_agents_openai.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    agent_name=agent_name,
    requirements_file="requirements.txt",
    region=region,
)
response

### AgentCore Runtime에 에이전트 시작

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime에 시작하겠습니다. 이 과정에서 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

#### 자동 생성된 역할에 필요한 추가 정책 연결

에이전트에 Outbound Auth를 추가하므로 자동 생성된 역할에 기본으로 포함되지 않은 API key와 secret에 액세스해야 합니다. 이를 위해 자동 생성된 IAM 역할에 추가 권한을 부여합니다. 먼저 역할을 조회한 다음 필요한 권한을 추가하겠습니다.

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

runtime_response = agentcore_control_client.get_agent_runtime(agentRuntimeId=launch_result.agent_id)
runtime_role = runtime_response["roleArn"]

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "GetResourceAPIKey",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:GetResourceApiKey"],
            "Resource": "*",
        },
        {
            "Sid": "SecretManager",
            "Effect": "Allow",
            "Action": ["secretsmanager:GetSecretValue"],
            "Resource": "arn:aws:secretsmanager:*:*:secret:bedrock-agentcore*",
        },
    ],
}
iam_client = boto3.client("iam", region_name=region)

response = iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인하겠습니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### AgentCore Runtime 호출

이제 payload로 AgentCore Runtime을 호출할 수 있습니다.

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "Hello"}, user_id="userid_1234567890")
invoke_response

### 호출 결과 처리

이제 호출 결과를 처리하여 애플리케이션에 포함할 수 있습니다.

In [ ]:
from IPython.display import Markdown, display

response_text = invoke_response["response"][0]
display(Markdown(response_text))

### boto3로 AgentCore Runtime 호출

AgentCore Runtime이 생성되었으므로 모든 AWS SDK에서 호출할 수 있습니다. 예를 들어 boto3의 `invoke_agent_runtime` 메서드를 사용할 수 있습니다.

In [ ]:
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    runtimeUserId="userid_1234567890",
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How much is 2X2?"}),
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                logger.info(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## 정리(선택 사항)

이제 생성한 AgentCore Runtime을 정리하겠습니다.

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

iam_client = boto3.client("iam")

runtime_delete_response = agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

policies = iam_client.list_role_policies(RoleName=runtime_role.split("/")[1], MaxItems=100)

for policy_name in policies["PolicyNames"]:
    iam_client.delete_role_policy(RoleName=runtime_role.split("/")[1], PolicyName=policy_name)
iam_response = iam_client.delete_role(RoleName=runtime_role.split("/")[1])

# 축하합니다!